# Лабораторная лабора №3 - Проведение исследований с решающим деревом


### Подготовка

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import sklearn
import keras
import pandas as pd
import numpy as np
import re
import torch

from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim


from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.preprocessing import LabelEncoder

In [7]:
df = pd.read_excel("/content/drive/MyDrive/ai_course/dataset.xlsx")
df.to_csv("dataset.csv", index = False)
df

,oid,text,category
0,749208109,СПОЧНО СООБЩЕСТВО ПРОДАЕТСЯ ЗА 1300Р ЗА ПОКУПК...,esport
1,749208109,Пусть это побудет здесь БорьбаВпрямомЭфире How...,esport
2,749208109,Раздача пиздюлей от Мунсунга. HowToFtokenoid Б...,esport
3,749208109,Не знаю как вам но мне стилистика нравится пус...,esport
4,749208109,Скриншоты из новой главы. Тэхунчика показали и...,esport
...,...,...,...
53193,910636962,8 битная буря снова накрыла пикселями автомоби...,autosport
53194,669736851,Ира Сидоркова объясняет как сказалась на ее ма...,autosport
53195,558919241,24 я ракетка мира хорват Марин Чилич обыграл и...,tennis
53196,776944963,Стал известен календарь мужской сборной России...,volleyball


In [8]:
df_reg = pd.read_csv("/content/drive/MyDrive/ai_course/train.csv")
df_reg

,cow_id,milk_yield_kg,feed_energy_eke,feed_crude_protein_g,sugar_protein_ratio,breed,pasture_type,sire_breed,milk_fat_pct,milk_protein_pct,milk_taste_label,age_group
0,488,6005,13.5,1842,0.940,РефлешнСоверинг,Равнинное,Айдиал,3.62,3.073,не вкусно,более_2_лет
1,422,5982,13.8,1722,0.890,РефлешнСоверинг,Холмистое,Айдиал,3.61,3.073,не вкусно,более_2_лет
2,105,5700,14.4,1934,0.885,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.079,вкусно,более_2_лет
3,115,5412,12.1,1924,0.890,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.071,не вкусно,менее_2_лет
4,350,6171,15.3,1966,0.940,РефлешнСоверинг,Равнинное,Соверин,3.73,3.076,вкусно,более_2_лет
...,...,...,...,...,...,...,...,...,...,...,...,...
502,72,5809,13.5,2093,0.895,РефлешнСоверинг,Равнинные,Айдиалл,3.61,3.079,не вкусно,более_2_лет
503,107,5417,13.2,1848,0.890,Вис Бик Айдиал,Холмистое,Соверин,3.57,3.076,вкусно,менее_2_лет
504,271,6597,14.2,2204,0.950,РефлешнСоверинг,Равнинное,Соверин,3.74,3.079,не вкусно,более_2_лет
505,436,6054,15.7,1859,0.940,РефлешнСоверинг,Холмистое,Соверин,3.71,3.080,вкусно,более_2_лет


## 2. Создание бейзлайна и оценка качества

### Бейзлайн классификация

In [36]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53198 entries, 0 to 53197
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   oid       53198 non-null  int64 
 1   text      53198 non-null  object
 2   category  53198 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [38]:
RANDOM_STATE = 42

TEXT_COL = "text"
LABEL_COL = "category"

In [39]:
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
df

,oid,text,category
0,749208109,СПОЧНО СООБЩЕСТВО ПРОДАЕТСЯ ЗА 1300Р ЗА ПОКУПК...,esport
1,749208109,Пусть это побудет здесь БорьбаВпрямомЭфире How...,esport
2,749208109,Раздача пиздюлей от Мунсунга. HowToFtokenoid Б...,esport
3,749208109,Не знаю как вам но мне стилистика нравится пус...,esport
4,749208109,Скриншоты из новой главы. Тэхунчика показали и...,esport
...,...,...,...
53193,910636962,8 битная буря снова накрыла пикселями автомоби...,autosport
53194,669736851,Ира Сидоркова объясняет как сказалась на ее ма...,autosport
53195,558919241,24 я ракетка мира хорват Марин Чилич обыграл и...,tennis
53196,776944963,Стал известен календарь мужской сборной России...,volleyball


In [40]:
le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL].astype(str))
X = df[TEXT_COL].astype(str)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [12]:
classification = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=80_000,
        sublinear_tf=True,
        norm="l2"
    )),
    ("dt", DecisionTreeClassifier(
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1
    ))
])

In [15]:
classification.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=80000, min_df=2,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('dt', DecisionTreeClassifier(random_state=42))])

In [15]:
y_pred = classification.predict(X_test)

In [16]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

   athletics       0.80      0.80      0.80       957
   autosport       0.70      0.66      0.68       630
  basketball       0.70      0.65      0.67       866
  boardgames       0.84      0.83      0.84       910
      esport       0.52      0.58      0.55       902
     extreme       0.47      0.49      0.48       634
    football       0.49      0.51      0.50       852
      hockey       0.51      0.43      0.47       370
martial_arts       0.56      0.60      0.58       852
   motosport       0.82      0.81      0.82       906
      tennis       0.82      0.79      0.80       917
  volleyball       0.69      0.70      0.70       937
winter_sport       0.65      0.64      0.64       907

    accuracy                           0.67     10640
   macro avg       0.66      0.65      0.65     10640
weighted avg       0.67      0.67      0.67     10640



In [17]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
cm_df

,athletics,autosport,basketball,boardgames,esport,extreme,football,hockey,martial_arts,motosport,tennis,volleyball,winter_sport
athletics,761,6,16,10,32,24,14,9,29,7,2,18,29
autosport,14,414,11,9,37,24,20,5,27,24,9,8,28
basketball,17,11,561,10,51,21,69,26,35,11,10,30,14
boardgames,11,17,14,753,30,23,7,3,16,4,8,11,13
esport,20,15,35,25,527,60,68,17,46,12,25,24,28
extreme,33,14,10,19,49,310,32,7,57,34,7,23,39
football,11,17,53,11,57,44,433,35,59,15,30,57,30
hockey,5,2,19,7,30,11,49,160,16,8,11,28,24
martial_arts,22,21,18,10,55,53,48,13,509,10,18,29,46
motosport,8,27,17,5,29,31,8,9,16,736,3,7,10


#### Выводы по бейзлайну

Результат Weighted F1 = 0.67

Очень плохое качество. Лучшие результаты у классов boardgames, motosport, tennis (F1 ≈ 0.8–0.84)

Базовое решающее дерево дало результат хуже, чем логистическая регрессия

Причины:

- дерево легко переобучается на большом количестве признаков TF-IDF,

- не имеет регуляризации (глубина не ограничена),

- не учитывает взаимосвязи между похожими словами.

Для улучшения нужно добавить ограничения на глубину, минимальное число объектов в листьях и провести настройку гиперпараметров

### Бейзлайн регрессия

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
RANDOM_STATE=42

In [7]:
train = pd.read_csv("/content/drive/MyDrive/ai_course/train.csv")
test = pd.read_csv("/content/drive/MyDrive/ai_course/test.csv")
train.head()

,cow_id,milk_yield_kg,feed_energy_eke,feed_crude_protein_g,sugar_protein_ratio,breed,pasture_type,sire_breed,milk_fat_pct,milk_protein_pct,milk_taste_label,age_group
0,488,6005,13.5,1842,0.940,РефлешнСоверинг,Равнинное,Айдиал,3.62,3.073,не вкусно,более_2_лет
1,422,5982,13.8,1722,0.890,РефлешнСоверинг,Холмистое,Айдиал,3.61,3.073,не вкусно,более_2_лет
2,105,5700,14.4,1934,0.885,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.079,вкусно,более_2_лет
3,115,5412,12.1,1924,0.890,Вис Бик Айдиал,Равнинное,Соверин,3.57,3.071,не вкусно,менее_2_лет
4,350,6171,15.3,1966,0.940,РефлешнСоверинг,Равнинное,Соверин,3.73,3.076,вкусно,более_2_лет


In [8]:
TARGET_COL = "milk_yield_kg"
ID_COL_CANDIDATES = ["cow_id"]

In [9]:
feature_cols = [c for c in train.columns if c != TARGET_COL]

id_col = next((c for c in ID_COL_CANDIDATES if c in test.columns), None)
if id_col is None:
    test_id = pd.Series(test.index, name="row_id")
else:
    test_id = test[id_col].copy()

In [10]:
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

Минимальные препроцессинг данных для работы модели

In [11]:
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ],
    remainder="drop"
)

In [12]:
reg = Pipeline(steps=[
    ("prep", preprocess),
    ("dt", DecisionTreeRegressor(
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1
    ))
])

In [13]:
X = train[feature_cols]
y = train[TARGET_COL].astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [26]:
reg.fit(X_tr, y_tr)
y_hat = reg.predict(X_te)

mae  = mean_absolute_error(y_te, y_hat)
mse = mean_squared_error(y_te, y_hat)
r2 = r2_score(y_te, y_hat)
print(f"MAE : {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"R^2 : {r2:.4f}")

MAE : 562.5686
MSE: 15989883.9804
R^2 : -59.0249


#### Выводы по регрессии

Качество крайне низкое: отрицательный R2 означает, что модель предсказывает хуже, чем простое среднее значение.
Скорее всего, дерево полностью переобучилось

Для улучшения нужно ограничить глубину, добавить регуляризацию и проверить важность признаков.

## 3. Улучшение бейзлайна

### Классификация

Гипотезы для улучшения:

1. Ограничить глубину дерева и минимальное число образцов в листьях, чтобы уменьшить переобучение.

2. Попробовать на чистых данных

3. Добавить балансировку классов

Ожидание: F1-score и точность вырастут, особенно для редких классов.

#### Загрузка данных

Для деревьев решения эмбеддинги не очень хороший вариант (я проводила тест, метрика ~0.47)

Балансировка SMOTE в данном случае тоже может дать плохой результат, так как будет "портиться" семантика

Поэтому будем тут пробовать tf-idf + DecisionTreeClassifier с подбором гиперпараметров

In [30]:
# import numpy as np
# from joblib import load

# X_train_embeddings = np.load("/content/drive/MyDrive/ai_course/X_train_embeddings.npy")
# X_test_embeddings  = np.load("/content/drive/MyDrive/ai_course/X_test_embeddings.npy")
# y_train = np.load("/content/drive/MyDrive/ai_course/y_train.npy")
# y_test = np.load("/content/drive/MyDrive/ai_course/y_test.npy")

In [18]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.9 MB/s eta 0:00:00


#### Обучение модели

In [29]:
RANDOM_STATE = 42

In [31]:
import optuna
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score

In [32]:
def make_pipeline(trial):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.9,
            max_features=30_000,
            sublinear_tf=True,
            norm="l2",
        )),
        ("dt", DecisionTreeClassifier(
            criterion=trial.suggest_categorical("criterion", ["gini", "entropy"]),
            max_depth=trial.suggest_int("max_depth", 4, 30),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            random_state=RANDOM_STATE
        ))
    ])

In [33]:
def objective(trial):
    pipe = make_pipeline(trial)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
    )
    return scores.mean()

In [23]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

[I 2025-12-12 08:33:25,088] A new study created in memory with name: no-name-2efc077b-a4b1-489c-b7c3-d13a28a14983


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2025-12-12 08:34:12,882] Trial 0 finished with value: 0.3997409520534941 and parameters: {'criterion': 'entropy', 'max_depth': 23, 'min_samples_split': 20, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.3997409520534941.
[I 2025-12-12 08:34:37,614] Trial 1 finished with value: 0.16034209250856743 and parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.3997409520534941.
[I 2025-12-12 08:35:09,395] Trial 2 finished with value: 0.338157359344411 and parameters: {'criterion': 'gini', 'max_depth': 20, 'min_samples_split': 17, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.3997409520534941.
[I 2025-12-12 08:35:43,931] Trial 3 finished with value: 0.34021779226548604 and parameters: {'criterion': 'gini', 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.3997409520534941.
[I 2025-12-12 08:36:32,323] Trial 4 finished with value: 0.376750951899069 and paramet

In [24]:
print("Best params:", study.best_trial.params)
print("Best f1_macro:", round(study.best_value, 4))

Best params: {'criterion': 'entropy', 'max_depth': 30, 'min_samples_split': 15, 'min_samples_leaf': 7}
Best f1_macro: 0.4382


In [25]:
best_pipe = make_pipeline(study.best_trial)
best_pipe.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, max_features=30000, min_df=3,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('dt',
                 DecisionTreeClassifier(criterion='entropy', max_depth=30,
                                        min_samples_leaf=7,
                                        min_samples_split=15,
                                        random_state=42))])

In [26]:
y_pred = best_pipe.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.53      0.66       957
           1       0.78      0.35      0.49       630
           2       0.88      0.28      0.43       866
           3       0.66      0.56      0.60       910
           4       0.19      0.23      0.21       902
           5       0.22      0.26      0.24       634
           6       0.13      0.69      0.22       852
           7       0.66      0.19      0.29       370
           8       0.88      0.29      0.43       852
           9       0.90      0.63      0.74       906
          10       0.85      0.54      0.66       917
          11       0.73      0.38      0.50       937
          12       0.47      0.18      0.26       907

    accuracy                           0.41     10640
   macro avg       0.63      0.39      0.44     10640
weighted avg       0.64      0.41      0.46     10640



#### ИТОГ 3 пункта

Метрики на бейзлайне
```
accuracy 0.68 10640
macro avg 0.67 0.66 0.66 10640
weighted avg 0.68 0.68 0.68 10640
```

пробовала на эмбеддингах и tf-idf, тоже плохо

Возможно, причина в слишуом маленьком max_depth и небольшом кол-во параметров

DecisionTree, в целом, очень чуствительна к шуму и признакам, с подобными задачами обычно лучше справляются LogisticRegression или SVC

### Регрессия

Попробуем:

1. Ограничить глубину дерева и размер листа, чтобы избежать переобучения.

2. Использовать лог-трансформацию таргета для сглаживания выбросов.

Ожидание: Ошибка (MAE, MSE) снизится

#### Загрузка чистых данных

In [14]:
X_test = pd.read_csv("/content/drive/MyDrive/ai_course/X_test.csv")
X_train = pd.read_csv("/content/drive/MyDrive/ai_course/X_train.csv")
y_train = pd.read_csv("/content/drive/MyDrive/ai_course/y_train.csv").squeeze()
y_test  = pd.read_csv("/content/drive/MyDrive/ai_course/y_test.csv").squeeze()

#### Обучение

In [15]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.1 MB/s eta 0:00:00


In [16]:
RANDOM_STATE=42

In [17]:
import numpy as np
import optuna
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

In [18]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop",
)

In [19]:
base_reg = Pipeline(steps=[
    ("prep", preprocess),
    ("dt", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [22]:
def objective_reg(trial):
    hp = {
        "criterion": trial.suggest_categorical(
            "criterion", ["squared_error", "friedman_mse", "absolute_error"]
        ),
        "max_depth": trial.suggest_int("max_depth", 4, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 40),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 0.01),
    }

    if hp["min_samples_split"] < 2 * hp["min_samples_leaf"]:
        hp["min_samples_split"] = 2 * hp["min_samples_leaf"]

    params_prefixed = {f"dt__{k}": v for k, v in hp.items()}
    reg = clone(base_reg)
    reg.set_params(**params_prefixed)

    scores = cross_val_score(
        reg,
        X_train, y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
        error_score="raise",
    )
    return scores.mean()

In [23]:
study_reg = optuna.create_study(direction="maximize")
study_reg.optimize(objective_reg, n_trials=60, show_progress_bar=True)

print("Best params:", study_reg.best_trial.params)
print("Best MegMAE:", round(study_reg.best_value, 4))

[I 2025-12-14 10:33:11,688] A new study created in memory with name: no-name-8878d85b-95ad-4359-a7ee-ad2c7d340979


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2025-12-14 10:33:15,481] Trial 0 finished with value: -199.04771604938273 and parameters: {'criterion': 'absolute_error', 'max_depth': 22, 'min_samples_split': 34, 'min_samples_leaf': 9, 'max_features': 'log2', 'ccp_alpha': 0.007541238803212036}. Best is trial 0 with value: -199.04771604938273.
[I 2025-12-14 10:33:15,627] Trial 1 finished with value: -198.49422839506173 and parameters: {'criterion': 'absolute_error', 'max_depth': 12, 'min_samples_split': 39, 'min_samples_leaf': 4, 'max_features': 'log2', 'ccp_alpha': 0.003679075541787522}. Best is trial 1 with value: -198.49422839506173.
[I 2025-12-14 10:33:15,781] Trial 2 finished with value: -140.02279598205953 and parameters: {'criterion': 'squared_error', 'max_depth': 25, 'min_samples_split': 22, 'min_samples_leaf': 18, 'max_features': None, 'ccp_alpha': 0.00707117881579944}. Best is trial 2 with value: -140.02279598205953.
[I 2025-12-14 10:33:15,921] Trial 3 finished with value: -200.7998270715128 and parameters: {'criterion': 

In [24]:
best_reg = clone(base_reg)
best_reg.set_params(**{f"dt__{k}": v for k, v in study_reg.best_trial.params.items()})
best_reg.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['feed_energy_eke',
                                                   'feed_crude_protein_g',
                                                   'sugar_protein_ratio',
                                                   'milk_fat_pct',
                                                   'milk_protein_pct']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['breed', 'pasture_type',
                                                   'sire_breed',
                                                   'milk_taste_label',
                                                   'age_group'])])),
                ('dt',
                 DecisionTreeRegressor(ccp_alpha=0.0018921046875930597,
                                       max_depth=16, min_samples_leaf=11,
                                       min_samples_split=18,
                                       random_state=42))])

In [25]:
y_pred = best_reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE : {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"R^2 : {r2:.4f}")

MAE : 131.5430
MSE: 31747.7777
R^2 : 0.8664


#### ИТОГ РЕГРЕССИЯ


Напоминаю старые метрики

```
MAE : 562.5686
MSE: 15989883.9804
R^2 : -59.0249
```




В бейзлайне дерево практически не справлялось с задачей, сейчас - после очистки и подбора парамтеров получаем очень хорошие метрики. Модель лучшилось в 4 раза

C помощью Optuna легко и быстро получилось подобрать лучшие параметры

## 4. Имплементация алгоритма машинного обучения

### Классификация

Гипотеза: Своя логистическая регрессия (softmax) с L2-регуляризацией и mini-batch SGD при тех же признаках даст качество, сопоставимое со sklearn

In [9]:
import numpy as np
from joblib import load

X_train_embeddings = np.load("/content/drive/MyDrive/ai_course/X_train_embeddings.npy")
X_test_embeddings  = np.load("/content/drive/MyDrive/ai_course/X_test_embeddings.npy")
y_train = np.load("/content/drive/MyDrive/ai_course/y_train.npy")
y_test = np.load("/content/drive/MyDrive/ai_course/y_test.npy")

In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

class MyDecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, task=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.task = task
        self.tree = None

    def _gini(self, y):
        if len(y) == 0:
            return 0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def _mse(self, y):
        if len(y) == 0:
            return 0
        return np.mean((y - np.mean(y)) ** 2)

    def _split(self, X, y, feature_idx, threshold):
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        return X[left_mask], y[left_mask], X[right_mask], y[right_mask]

    def _best_split(self, X, y):
        best_loss = float('inf')
        best_split = None
        n_samples, n_features = X.shape
        if n_samples < self.min_samples_split:
            return None

        if self.task == 'classification':
            base_loss = self._gini(y)
        else:
            base_loss = self._mse(y)

        for feat in range(n_features):
            thresholds = np.unique(X[:, feat])
            for th in thresholds:
                X_left, y_left, X_right, y_right = self._split(X, y, feat, th)
                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                if self.task == 'classification':
                    loss = (len(y_left) / len(y)) * self._gini(y_left) + \
                           (len(y_right) / len(y)) * self._gini(y_right)
                else:
                    loss = (len(y_left) / len(y)) * self._mse(y_left) + \
                           (len(y_right) / len(y)) * self._mse(y_right)

                if loss < best_loss:
                    best_loss = loss
                    best_split = {
                        'feature': feat,
                        'threshold': th,
                        'left_X': X_left,
                        'left_y': y_left,
                        'right_X': X_right,
                        'right_y': y_right
                    }

        if best_split is None or best_loss >= base_loss:
            return None
        return best_split

    def _build_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y)) == 1:
            if self.task == 'classification':
                values, counts = np.unique(y, return_counts=True)
                return {'leaf': True, 'value': values[np.argmax(counts)]}
            else:
                return {'leaf': True, 'value': np.mean(y)}

        split = self._best_split(X, y)
        if split is None:
            if self.task == 'classification':
                values, counts = np.unique(y, return_counts=True)
                return {'leaf': True, 'value': values[np.argmax(counts)]}
            else:
                return {'leaf': True, 'value': np.mean(y)}

        left_subtree = self._build_tree(split['left_X'], split['left_y'], depth + 1)
        right_subtree = self._build_tree(split['right_X'], split['right_y'], depth + 1)

        return {
            'leaf': False,
            'feature': split['feature'],
            'threshold': split['threshold'],
            'left': left_subtree,
            'right': right_subtree
        }

    def fit(self, X, y):
        if self.task is None:
            self.task = 'regression' if (np.issubdtype(y.dtype, np.number) and len(np.unique(y)) > 20) else 'classification'
        self.tree = self._build_tree(X, y)

    def _predict_sample(self, x, tree):
        if tree['leaf']:
            return tree['value']
        if x[tree['feature']] <= tree['threshold']:
            return self._predict_sample(x, tree['left'])
        else:
            return self._predict_sample(x, tree['right'])

    def predict(self, X):
        return np.array([self._predict_sample(x, self.tree) for x in X])
    def evaluate(self, X, y_true):
        y_pred = self.predict(X)
        if self.task == 'classification':
            acc = accuracy_score(y_true, y_pred)
            report = classification_report(y_true, y_pred, zero_division=0)
            return acc, report
        else:
            mse = mean_squared_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)
            report = f"MSE = {mse:.6f}\n  R2 = {r2:.6f}"
            return r2, report

In [13]:
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler

MAX_PER_CLASS = 1500

class_counts = pd.Series(y_train).value_counts()

sampling_strategy = {
    cls: min(cnt, MAX_PER_CLASS)
    for cls, cnt in class_counts.items()
}

rus = RandomUnderSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_dummy = np.arange(len(y_train)).reshape(-1, 1)

idx_resampled, y_train_resampled = rus.fit_resample(X_dummy, y_train)
idx_resampled = idx_resampled.flatten()

In [14]:
X_train_resampled = X_train_embeddings[idx_resampled]

In [15]:
print(X_train_resampled.shape)
print(y_train_resampled.shape)

(16026, 768)
(16026,)


In [ ]:
dt_emb = MyDecisionTree(max_depth=3, min_samples_split=3)
dt_emb.fit(X_train_resampled, y_train_resampled)

In [ ]:
acc, report = dt_emb.evaluate(X_test, y_test)
print(report)

#### ИТОГ 4 ПУНКТ


Я обрезала датасет, так как он очень большой и слишком много времени занимает его обучение

### Регрессия

Гипотезы: наша модель должна давать близкие метрики к sklearn и сильно лучше бейзлайна

In [26]:
import numpy as np

class SimpleTreeNode:
    def __init__(self, feature_index=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class SimpleDecisionTreeRegressor:
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root_ = None

    def _mse(self, y):
        if y.size == 0:
            return 0.0
        mu = y.mean()
        return np.mean((y - mu) ** 2)

    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        if n_samples < self.min_samples_split:
            return None, None

        best_feature = None
        best_threshold = None
        best_loss = np.inf

        parent_mse = self._mse(y)

        for j in range(n_features):
            xj = X[:, j]
            # сортировка
            order = np.argsort(xj)
            xj_sorted = xj[order]
            y_sorted = y[order]

            # кандидаты порогов
            unique_vals = np.unique(xj_sorted)
            if unique_vals.size == 1:
                continue

            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0

            for thr in thresholds:
                left_mask = xj_sorted <= thr
                right_mask = ~left_mask

                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue

                y_left = y_sorted[left_mask]
                y_right = y_sorted[right_mask]

                loss = (y_left.size * self._mse(y_left) +
                        y_right.size * self._mse(y_right)) / n_samples

                if loss < best_loss:
                    best_loss = loss
                    best_feature = j
                    best_threshold = thr

        if best_feature is None or best_loss >= parent_mse:
            return None, None

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth):
        # условия остановки
        if (depth >= self.max_depth or
            y.size < self.min_samples_split or
            np.unique(y).size == 1):
            return SimpleTreeNode(value=y.mean())

        feature_index, threshold = self._best_split(X, y)
        if feature_index is None:
            return SimpleTreeNode(value=y.mean())

        # делим выборку
        left_mask = X[:, feature_index] <= threshold
        right_mask = ~left_mask

        X_left, y_left = X[left_mask], y[left_mask]
        X_right, y_right = X[right_mask], y[right_mask]

        if y_left.size < self.min_samples_leaf or y_right.size < self.min_samples_leaf:
            return SimpleTreeNode(value=y.mean())

        left_child = self._build_tree(X_left, y_left, depth + 1)
        right_child = self._build_tree(X_right, y_right, depth + 1)

        return SimpleTreeNode(
            feature_index=feature_index,
            threshold=threshold,
            left=left_child,
            right=right_child,
            value=None
        )

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.root_ = self._build_tree(X, y, depth=0)
        return self

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(x, self.root_) for x in X])


In [27]:
X_train_enc = preprocess.fit_transform(X_train)
X_test_enc  = preprocess.transform(X_test)

X_train_arr = X_train_enc.toarray() if hasattr(X_train_enc, "toarray") else X_train_enc
X_test_arr  = X_test_enc.toarray() if hasattr(X_test_enc, "toarray") else X_test_enc

y_train_arr = np.asarray(y_train, dtype=float)
y_test_arr  = np.asarray(y_test, dtype=float)

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tree_np = SimpleDecisionTreeRegressor(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
)

tree_np.fit(X_train_arr, y_train_arr)
y_pred_np = tree_np.predict(X_test_arr)

In [29]:
mae_np = mean_absolute_error(y_test_arr, y_pred_np)
mse_np = mean_squared_error(y_test_arr, y_pred_np)
r2_np = r2_score(y_test_arr, y_pred_np)

print(f"MAE : {mae_np:.4f}")
print(f"MSE: {mse_np:.4f}")
print(f"R^2 : {r2_np:.4f}")

MAE : 138.1260
MSE: 33771.6842
R^2 : 0.8578


#### ИТОГ РЕГРЕССИЯ

Результат, как и ожидалось, получился близкий к sklearn, но чуть хуже, при этом сильно лучше бейзлайна, что логично, так как у нас очищенные данные